# 10 — Benchmark du retrieval

Ce notebook mesure le retrieval indépendamment du LLM. Les questions de démonstration doivent être remplacées par des questions relues et rattachées à des pages officielles.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

In [ ]:
from morocco_legal_rag.chunking import chunk_page
from morocco_legal_rag.evaluation import EvaluationQuestion, evaluate_retriever
from morocco_legal_rag.retrieval import HybridRetriever

examples = {
    "A": "Le délai pédagogique de traitement est de dix jours ouvrables.",
    "B": "Une correction nécessite une demande écrite et un justificatif rectifié.",
    "C": "Le dépôt pédagogique est disponible au guichet de démonstration.",
}
chunks = [
    chunk_page(document_id=key, title=f"Démo {key}", page=1, text=text, size=40, overlap=5)[0]
    for key, text in examples.items()
]
retriever = HybridRetriever()
retriever.fit(chunks)
questions = [
    EvaluationQuestion("q1", "délai de traitement", frozenset({"A"})),
    EvaluationQuestion("q2", "corriger avec une demande écrite", frozenset({"B"})),
    EvaluationQuestion("q3", "dépôt au guichet", frozenset({"C"})),
]
evaluate_retriever(retriever, questions, k=2)

## Expériences à enregistrer

- BM25 seul, dense seul et hybride ;
- plusieurs tailles de chunks et overlaps ;
- hashing baseline contre multilingual-e5 ;
- avec et sans filtres de métadonnées ;
- avec et sans reranker.

Ne retenir une amélioration que si le benchmark progresse sans dégrader les questions sans réponse.